In [14]:
import numpy as np
import pandas as pd
import re
import math
import time
import matplotlib.pyplot as plt
%matplotlib inline

In [15]:
FILENAME="input.txt"
# FILENAME="test.txt"
DIAG=False
DIAG2 = False
VISUALISE=False
INF_NEG_CONST = -np.inf
INF_POS_CONST = np.inf


In [16]:
def parseFile(file_name):
    with open(file_name) as file:
        lines = file.readlines()
    
    
    brick_cells = {}
    brick_detail = {INF_NEG_CONST:{"height":1},
                    INF_POS_CONST:{"height":1},
                   }
    
    
    for i,line in enumerate(lines):
        if line[-1] == "\n": line = line[:-1]
        str_re = "(\d+),(\d+),(\d+)~(\d+),(\d+),(\d+)"
        mmm = re.match(str_re,line)
        mybrick_st = np.array((int(mmm[1]),int(mmm[2]),int(mmm[3])))
        mybrick_en = np.array((int(mmm[4]),int(mmm[5]),int(mmm[6])))
        
        mybrick_cells = [tuple(x.tolist()) for x in np.linspace(mybrick_en,mybrick_st,int(np.linalg.norm(mybrick_st-mybrick_en))+1).astype(int)]
        mybrick_size_z = mybrick_en[2]-mybrick_st[2]+1
        
        mybrick_len = mybrick_en-mybrick_st
        
        brick_cells.update(dict.fromkeys(mybrick_cells,i))
        brick_detail[i]={"height":mybrick_size_z}
    
    return brick_cells, brick_detail
    

In [17]:
brick_cells, brick_detail = parseFile(FILENAME)


In [18]:
# find dependencies between and create a graph
cells_from_bottom=sorted(list(brick_cells.keys()),key=lambda x:x[2])
max_x, max_y, _ =  np.array(cells_from_bottom).max(axis=0)
lastseen_layer = np.ones((max_x+1,max_y+1))*INF_NEG_CONST

graph_edges = []

# 1. Parse existing cells into graph nodes and edges
for x,y,z in cells_from_bottom:
    i=brick_cells[(x,y,z)]
    prev_i = lastseen_layer[x,y]
    if prev_i != i and prev_i != INF_NEG_CONST : graph_edges += [(int(prev_i),int(i),-brick_detail[prev_i]["height"])]
    lastseen_layer[x,y]=i

# 2. Add start and end node
root_nodes = {x[0] for x in graph_edges} - {x[1] for x in graph_edges}
leaf_nodes = {x[1] for x in graph_edges} - {x[0] for x in graph_edges}
for x in root_nodes: graph_edges += [(INF_NEG_CONST,int(x),-brick_detail[INF_NEG_CONST]["height"])]
for x in leaf_nodes: graph_edges += [(int(x),INF_POS_CONST,-brick_detail[x]["height"])]



In [19]:
# 3. Construct graph
import networkx as nx
G = nx.DiGraph()
G.add_weighted_edges_from(graph_edges)
print(G)
if VISUALISE: nx.draw(G, with_labels=True, font_weight='bold')


DiGraph with 1390 nodes and 3224 edges


In [20]:
weights__a=[]
for x,y in G.edges():
    weights__a += [G.get_edge_data(x,y)["weight"]]
print(max(weights__a),min(weights__a))

-1 -5


In [21]:
# 4. Calculate longest path from source (shortest path in graph as lenghts of edges flipped)
neg_heights = nx.shortest_path_length(G,source=INF_NEG_CONST,weight=(lambda x,y,z:z['weight']),method="bellman-ford")

for k,v in neg_heights.items():
    brick_detail[k]["row_start"]=-v+1


In [22]:
print(G)

DiGraph with 1390 nodes and 3224 edges


In [23]:
# 5. Eliminate edges which don't matter for support
G_lean = G.copy()
to_remove = []
for x_n,y_n in G_lean.edges():
    x_row_st, y_row_st = brick_detail[x_n]["row_start"],brick_detail[y_n]["row_start"]
    x_height = brick_detail[x_n]["height"]
    if x_row_st+x_height != y_row_st:
        to_remove+= [(x_n,y_n)]
        
for x,y in to_remove:
    G_lean.remove_edge(x,y)

#G_lean.remove_node(INF_NEG_CONST)
G_lean.remove_node(INF_POS_CONST)


In [24]:
if VISUALISE: nx.draw(G_lean,with_labels=True, font_weight='bold')
print(G_lean)

DiGraph with 1389 nodes and 1502 edges


In [25]:
# Part A - number of nodes whose all children have another parent
def getResA(G_lean):
    count_removable = 0

    nodes_nonremovable = []
    for node_x in G_lean.nodes:    
        if node_x == INF_NEG_CONST: continue

        if G_lean.out_degree(node_x) == 0: all_supported = True
        else:
            all_supported = (min([G_lean.in_degree(z) for z in G_lean.neighbors(node_x)]) > 1)

        # count_removable+=all_supported
        if all_supported: count_removable +=1
        else: nodes_nonremovable+=[node_x]
    
    return count_removable, nodes_nonremovable
        
%time resA, nodes_nonremovable = getResA(G_lean)
print("\nResult A:",resA)

CPU times: total: 0 ns
Wall time: 42 ms

Result A: 459


In [26]:
# Part B - remove each node and find unreachable nodes after doing that with in_df
def getResB(G_lean):
    count_dropped = 0
    all_nodes = set(G_lean.nodes())
    for node_x in nodes_nonremovable:
        G_lean_copy = G_lean.copy()
        G_lean_copy.remove_node(node_x)
        pppaths = nx.shortest_path_length(G_lean_copy,source=INF_NEG_CONST,weight=(lambda x,y,z:z['weight']),method="bellman-ford")
        unreachable = all_nodes - set(pppaths.keys())
        
        count_dropped += len(unreachable)-1
    return count_dropped

%time resB=getResB(G_lean)
print("\nResult B:",resB)

CPU times: total: 30.6 s
Wall time: 32.6 s

Result B: 75784
